# logit-thermo 探索：比熱の山と T_0（温度計の材料を取り出す、CPU）

- **必要な secret：なし**。**ランタイム：CPU（標準）**。見込みは 2〜5 分。
- 探索（予想なし）。A2-2・A2-5・A2-6 の 25 本の保存ファイルから、人の文章を読ませたときの「温度計の材料」`tcal`（記事ごと・61 の温度ごとの Σ(⟨z⟩_T − z_y)）だけを取り出して、
  小さなファイル `A2_explore/tcal_all.npz` にまとめます（Claude が手元で比熱を計算するため。元のファイルは 10MB を超えて Claude が直接取り込めない）。**結果は表示しません。**
- **通知**：🚀・🎉・⚠️。名札は「logit-thermo 探索 比熱の山（温度計の材料を取り出す、CPU）」。終わったら、またはエラーで止まったら、ランタイムを切ります。


In [ ]:
# セル1：通知、GPU、Drive
import os, time, urllib.request
NTFY_TOPIC = 'YOUR_NTFY_TOPIC'
NB_LABEL = 'logit-thermo 探索 比熱の山（温度計の材料を取り出す、CPU）'
def notify(msg, tags='pig'):
    try:
        req = urllib.request.Request('https://ntfy.sh/' + NTFY_TOPIC, data=(msg + '\n[' + NB_LABEL + ']').encode('utf-8'),
                                     headers={'Title': 'logit-thermo - explore heat capacity extract', 'Tags': tags})
        urllib.request.urlopen(req, timeout=10).read()
        print('通知を送りました：', msg)
    except Exception as e:
        print('通知は送れませんでした（実験には影響しません）：', type(e).__name__)
_NOTIFIED = {'error': False}
def _on_error(result):
    if result.error_in_exec is not None and not _NOTIFIED['error']:
        _NOTIFIED['error'] = True
        import traceback
        e = result.error_in_exec
        txt = ''.join(traceback.format_exception(type(e), e, e.__traceback__))
        v = os.environ.get('HF_TOKEN')
        if v: txt = txt.replace(v, '***')
        txt = txt.encode('utf-8')[-3300:].decode('utf-8', 'ignore')
        notify('⚠️ 止まりました（エラー）：\n' + txt, tags='warning')
if not globals().get('_HOOK_ON'):
    get_ipython().events.register('post_run_cell', _on_error); _HOOK_ON = True
notify('🚀 スタートしました（これが届けば通知の設定は OK です）', tags='rocket')

from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# セル2：取り出して保存（結果は表示しない）
import numpy as np, os
RUNS = {'A2_2': ['pythia-1b', 'qwen2.5-0.5b', 'qwen2.5-0.5b-inst', 'qwen2.5-1.5b', 'olmo2-1b', 'smollm2-1.7b', 'smollm2-1.7b-inst'],
        'A2_5': ['pythia-70m', 'pythia-160m', 'pythia-410m', 'pythia-1b', 'pythia-1.4b', 'pythia-2.8b',
                 'pythia-1b@step8000', 'pythia-1b@step24000', 'pythia-1b@step64000'],
        'A2_6': ['qwen2.5-0.5b', 'qwen2.5-1.5b', 'qwen2.5-3b', 'smollm2-135m', 'smollm2-360m', 'smollm2-1.7b',
                 'qwen3-0.6b', 'qwen3-1.7b', 'qwen3-4b']}
try:
    out, missing = {}, []
    for folder, names in RUNS.items():
        for n in names:
            p = '/content/drive/MyDrive/logit-thermo' + f'/{folder}/{n}.npz'
            if not os.path.exists(p):
                missing.append(p); continue
            Z = np.load(p)
            if 'tcal' not in Z.files:
                missing.append(p + ' (tcal なし)'); continue
            out[f'{folder}/{n}|tcal'] = Z['tcal'].astype(np.float64)
            out[f'{folder}/{n}|tcal_T'] = Z['tcal_T'].astype(np.float64)
    np.savez_compressed('/content/drive/MyDrive/logit-thermo' + '/A2_explore/tcal_all.npz', missing=np.array(missing), **out)
    print('保存しました：', len(out) // 2, '本。見つからなかったもの：', len(missing))
except Exception as e:
    import traceback
    _NOTIFIED['error'] = True
    txt = ''.join(traceback.format_exception(type(e), e, e.__traceback__))
    v = os.environ.get('HF_TOKEN')
    if v: txt = txt.replace(v, '***')
    txt = txt.encode('utf-8')[-3300:].decode('utf-8', 'ignore')
    notify('⚠️ 探索 比熱の山（温度計の材料を取り出す）で止まりました。ランタイムを切ります：\n' + txt, tags='warning')
    if globals().get('AUTO_DISCONNECT', True):
        from google.colab import drive as _d, runtime as _r
        _d.flush_and_unmount(); time.sleep(3); _r.unassign()
    raise


In [ ]:
# 最後：終わりの通知と、ランタイムを切る
notify('🎉 終わりました：温度計の材料を A2_explore/tcal_all.npz に保存しました', tags='tada')
if globals().get('AUTO_DISCONNECT', True):
    from google.colab import drive as _drive, runtime as _runtime
    _drive.flush_and_unmount(); time.sleep(3); _runtime.unassign()
